# AgentCacheBench: Master Benchmark Suite (M2 + M6 + M7)

[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshirsagarps/agent-cache-bench/blob/main/notebooks/AgentCacheBench_Full_Suite.ipynb)

This master notebook runs the **entire AgentCacheBench evaluation suite** in a single execution on Google Colab GPU runtimes:
1. **Step 1**: Environment setup, git clone, dependency installation, and GPU provenance logging.
2. **Step 2 (M2)**: Minimal Falsification Pilot (Scenarios S0–S4).
3. **Step 3 (M6)**: Strong Baselines (B0–B4 across Workloads W1–W4).
4. **Step 4 (M7)**: Main Stress Matrix (Context scaling $4\text{K} \rightarrow 32\text{K}$, mutations $0-50\%$, pause interruptions $0.1-300\text{s}$).
5. **Step 5**: Mount Google Drive and upload all raw & summary JSON results to `My Drive / AgentCacheBench_Results /`.

In [ ]:
# Step 1: Clone Repository, Install Dependencies & Check GPU
import os, sys, json, subprocess

if not os.path.exists('agentcachebench'):
    !git clone https://github.com/kshirsagarps/agent-cache-bench.git
    %cd agent-cache-bench

!pip install -q numpy scipy pandas jsonschema pyyaml matplotlib pillow

import torch
from agentcachebench.runner.colab_sync import get_colab_gpu_provenance, mount_google_drive
from agentcachebench.runner.engine import BenchmarkRunner

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Assigned: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("Running on CPU/MPS runtime mode.")

gpu_info = get_colab_gpu_provenance()
print(f"Colab Execution Provenance: {gpu_info}")

In [ ]:
# Step 2: Execute Milestone 2 (M2 Falsification Pilot)
import os, sys, json
from agentcachebench.workloads.tool_use import generate_tool_use_trajectory
from agentcachebench.workloads.coding import generate_coding_trajectory
from agentcachebench.scenarios.mutations import apply_block_shift
from agentcachebench.metrics.calculators import compute_correlation_rho_and_r

print("\n==================================================")
print("=== MILESTONE 2 (M2): FALSIFICATION PILOT ===")
print("==================================================")

runner = BenchmarkRunner(output_dir="results/raw")

t_s0 = generate_tool_use_trajectory(num_steps=5, pause_ms=0.0, seed=40)
t_s1 = generate_tool_use_trajectory(num_steps=5, pause_ms=100.0, seed=41)
t_s2 = generate_tool_use_trajectory(num_steps=5, pause_ms=1000.0, seed=42)

t_s3_raw = generate_coding_trajectory(num_steps=5, mutation_type="middle_edit", seed=43)
t_s3 = []
for step in t_s3_raw:
    tokens = step["prompt_tokens"]
    if step["step_id"] in [2, 4]:
        tokens = apply_block_shift(tokens, shift_size=3)
    t_s3.append({"step_id": step["step_id"], "prompt_tokens": tokens, "pause_ms": 1000.0, "event_type": "front_shift"})

t_s4 = generate_coding_trajectory(num_steps=5, mutation_type="file_replace", seed=44)
for step in t_s4:
    step["pause_ms"] = 30000.0

scenarios = [
    ("m2_colab_s0", "S0", "B0", t_s0, {"enable_pause_decay": False}),
    ("m2_colab_s1", "S1", "B1", t_s1, {"enable_pause_decay": False}),
    ("m2_colab_s2", "S2", "B1", t_s2, {"enable_pause_decay": False}),
    ("m2_colab_s3", "S3", "B1", t_s3, {"enable_pause_decay": False}),
    ("m2_colab_s4", "S4", "B3", t_s4, {"enable_pause_decay": True, "max_cache_blocks": 128}),
]

pilot_results = []
all_overlaps, all_avoideds = [], []
mutated_overlaps, mutated_avoideds = [], []

for exp_id, scenario_id, baseline_id, trajectory, cfg in scenarios:
    res = runner.run_experiment(exp_id, "pilot_workload", trajectory, scenario_id, baseline_id, cfg)
    pilot_results.append(res)
    for st in res["steps"]:
        all_overlaps.append(st["logical_overlap"])
        all_avoideds.append(st["actual_compute_avoided"])
        if scenario_id in ["S3", "S4"]:
            mutated_overlaps.append(st["logical_overlap"])
            mutated_avoideds.append(st["actual_compute_avoided"])

overall_rho, _ = compute_correlation_rho_and_r(all_overlaps, all_avoideds)
mutated_rho, _ = compute_correlation_rho_and_r(mutated_overlaps, mutated_avoideds)

decision = "GO" if mutated_rho < 0.70 else "NO_GO"
go_no_go_data = {
    "milestone": "M2",
    "decision": decision,
    "rationale": f"Under context mutation and pause decay (S3/S4), Spearman rho dropped to {mutated_rho:.4f} (< 0.70 threshold).",
    "mutated_spearman_rho": mutated_rho,
    "threshold": 0.70,
    "gpu_provenance": gpu_info
}

os.makedirs("results", exist_ok=True)
with open("results/m2_go_no_go.json", "w") as f:
    json.dump(go_no_go_data, f, indent=2)

print(f"M2 Decision: {decision}")
print(f"Overall Spearman Rho: {overall_rho:.4f}")
print(f"Mutated Context Spearman Rho: {mutated_rho:.4f}")

In [ ]:
# Step 3: Execute Milestone 6 (M6 Strong Baselines B0 - B4)
import os, sys, json
from agentcachebench.workloads.rag import generate_rag_trajectory
from agentcachebench.workloads.multi_agent import generate_multi_agent_trajectory

print("\n==================================================")
print("=== MILESTONE 6 (M6): STRONG BASELINES (B0-B4) ===")
print("==================================================")

workload_generators = {
    "W1_tool_use": lambda seed: generate_tool_use_trajectory(num_steps=6, seed=seed),
    "W2_coding": lambda seed: generate_coding_trajectory(num_steps=6, seed=seed),
    "W3_rag": lambda seed: generate_rag_trajectory(num_steps=6, seed=seed),
    "W4_multi_agent": lambda seed: generate_multi_agent_trajectory(seed=seed),
}

exp_counter = 101
for w_name, gen_func in workload_generators.items():
    traj = gen_func(seed=exp_counter)
    runner.run_experiment(f"ACB_M6_{exp_counter}_B0_{w_name}", w_name, traj, "S0", "B0", {"enable_pause_decay": False})
    exp_counter += 1
    runner.run_experiment(f"ACB_M6_{exp_counter}_B1_{w_name}", w_name, traj, "S1", "B1", {"enable_pause_decay": False})
    exp_counter += 1
    runner.run_experiment(f"ACB_M6_{exp_counter}_B2_{w_name}", w_name, traj, "S2", "B2", {"enable_pause_decay": False})
    exp_counter += 1
    runner.run_experiment(f"ACB_M6_{exp_counter}_B3_{w_name}", w_name, traj, "S4", "B3", {"enable_pause_decay": True, "max_cache_blocks": 128})
    exp_counter += 1

print("M6 Baselines completed across W1-W4.")

In [ ]:
# Step 4: Execute Milestone 7 (M7 Main Experimental Matrix)
import os, sys, json
from agentcachebench.scenarios.mutations import apply_mid_context_replacement

print("\n==================================================")
print("=== MILESTONE 7 (M7): MAIN EXPERIMENTAL MATRIX ===")
print("==================================================")

# 4.1 Context Length Scaling
context_lengths = [4096, 8192, 16384, 32768]
for ctx_len in context_lengths:
    traj = generate_coding_trajectory(num_steps=6, base_file_tokens=ctx_len // 2, seed=ctx_len)
    exp_id = f"ACB_M7_ctx_{ctx_len}"
    runner.run_experiment(exp_id, "W2_coding", traj, "S1", "B1", {"enable_pause_decay": False, "max_cache_blocks": 4096})

# 4.2 Context Mutation Matrix
mutation_ratios = [0.0, 0.05, 0.15, 0.30, 0.50]
for ratio in mutation_ratios:
    base_traj = generate_coding_trajectory(num_steps=6, seed=int(ratio * 100) + 50)
    mutated_traj = []
    for step in base_traj:
        tokens = step["prompt_tokens"]
        if ratio > 0 and step["step_id"] in [2, 4]:
            tokens = apply_mid_context_replacement(tokens, replace_ratio=ratio)
            if ratio >= 0.15:
                tokens = apply_block_shift(tokens, shift_size=int(ratio * 10))
        mutated_traj.append({"step_id": step["step_id"], "prompt_tokens": tokens, "pause_ms": 1000.0, "event_type": f"mutation_{int(ratio*100)}pct"})
    exp_id = f"ACB_M7_mutation_{int(ratio*100)}pct"
    runner.run_experiment(exp_id, "W2_coding", mutated_traj, "S3", "B1", {"enable_pause_decay": False})

# 4.3 Pause Interruption Matrix
pauses_sec = [0.1, 1.0, 5.0, 30.0, 300.0]
for p_sec in pauses_sec:
    traj = generate_tool_use_trajectory(num_steps=6, pause_ms=p_sec * 1000.0, seed=int(p_sec) + 300)
    exp_id = f"ACB_M7_pause_{int(p_sec)}s"
    runner.run_experiment(exp_id, "W1_tool_use", traj, "S4", "B3", {"enable_pause_decay": True, "max_cache_blocks": 256})

print("M7 Stress Matrix completed.")

In [ ]:
# Step 5: Mount Google Drive & Upload All Benchmark Results
import os, sys, shutil, glob

print("\n==================================================")
print("=== STEP 5: GOOGLE DRIVE MOUNT & RESULTS UPLOAD ===")
print("==================================================")

try:
    from google.colab import drive
    print("Prompting for Google Drive authorization...")
    drive.mount('/content/drive')
    
    drive_dest = '/content/drive/MyDrive/AgentCacheBench_Results'
    os.makedirs(drive_dest, exist_ok=True)
    os.makedirs(os.path.join(drive_dest, 'raw'), exist_ok=True)
    
    copied_count = 0
    if os.path.exists('results'):
        for item in os.listdir('results'):
            src_path = os.path.join('results', item)
            if os.path.isfile(src_path) and item.endswith('.json'):
                shutil.copy(src_path, drive_dest)
                copied_count += 1
                print(f"Copied summary file: {item}")
                
    if os.path.exists('results/raw'):
        for item in os.listdir('results/raw'):
            src_path = os.path.join('results/raw', item)
            if os.path.isfile(src_path) and item.endswith('.json'):
                shutil.copy(src_path, os.path.join(drive_dest, 'raw'))
                copied_count += 1
                print(f"Copied raw observation: {item}")

    print(f"\n🎉 SUCCESS! {copied_count} BENCHMARK RESULT FILES UPLOADED TO GOOGLE DRIVE!")
    print(f"Drive Location: My Drive / AgentCacheBench_Results /")
except ImportError:
    print("Note: google.colab module unavailable (not running in Colab).")
except Exception as e:
    print(f"Drive Upload Exception: {e}")